In [1]:
!pip install git+https://github.com/ibm-granite-community/utils \
    "langchain_community<0.3.0" \
    replicate

  Cloning https://github.com/ibm-granite-community/utils to /tmp/pip-req-build-pu1dhuzq
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/utils /tmp/pip-req-build-pu1dhuzq
  Resolved https://github.com/ibm-granite-community/utils to commit 97732c007b2768d5c61dc61169743373043252ec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from ibm_granite_community.notebook_utils import get_env_var
from langchain_community.llms import Replicate

model = Replicate(
    model="ibm-granite/granite-3.3-8b-instruct",
    replicate_api_token=get_env_var('REPLICATE_API_TOKEN'),
    model_kwargs={"max_tokens":1024, "temperature":0.2},
)

REPLICATE_API_TOKEN loaded from Google Colab secret.


In [3]:
def fewshot_prompt(context, question, book_titles, examples):
    """
    Creates a few-shot prompt for the model, where the model leverages examples to improve accuracy.

    Parameters:
    - context: str, contextual information for the prompt
    - question: str, specific question or task for the model to perform
    - book_titles: list, list of book titles to include in the prompt
    - examples: list of dicts, each containing 'question', 'context', and 'output' as keys to provide examples
    Returns:
    - str, the formatted few-shot prompt
    """

    # Format the book titles
    titles = ", ".join(book_titles)

    # Format the examples for the prompt
    formatted_examples = "\n\n".join(
        f"""
        Example {i+1}:
        User Question: {example['question']}
        Context: {example['context']}
        Model Output: {example['output']}
        """
        for i, example in enumerate(examples)
    )

    # Construct the few-shot prompt
    prompt = f"""
    You are an experienced programmer with 15 years of experience writing full-stack applications.
    Your task is to generate high-quality Python code for a Jupyter Notebook using ipywidgets UI components
    based on the provided context and user question.

    Here are some examples of similar tasks you have completed successfully:
    {formatted_examples}

    Now, using these examples as a reference, generate code for the following task:

    Context: {context}
    User Question: {question}
    Include descriptions of elements from the book titles: {titles}
    Ensure that:
    - The code is well-structured and uses consistent styling, coloring, and formatting for all UI elements.
    - Output only the Python code.
    """
    return prompt


In [4]:
# Function to get answer using few-shot prompting
def get_answer_using_fewshot(context, question, book_titles, examples):
    """
    Generates the response from the model based on a few-shot prompt.

    Parameters:
    - context: str, contextual information for the prompt
    - question: str, specific question for the model to answer
    - book_titles: list, list of book titles to include in the prompt

    Returns:
    - str, the generated result from the model
    """
    prompt = fewshot_prompt(context, question, book_titles, examples)
    result = model.invoke(prompt)

    return result

In [5]:
examples = [
{
    "question": "Add a styled header for my bookstore landing page",
    "context": "Gradient Background and Font Styling for Header",
    "output": """
import ipywidgets as widgets
from IPython.display import display

# Create a styled header
header = widgets.HTML(value="
    <div style='background: linear-gradient(to right, #6a11cb, #2575fc); padding: 20px; border-radius: 8px;'>
        <h1 style='color: white; text-align: center; font-family: Arial, sans-serif;'>"Welcome to Reader's Verse"</h1>
    </div>
")

display(header)
"""
},
{
    "question": "Enhance my bookstore catalog with hover effects",
    "context": "Interactive Book Tiles with Shadows and Hover Animation",
    "output": """
import ipywidgets as widgets
from IPython.display import display

# Create book tiles with hover effects
book_tiles = [
    widgets.HTML(value="
        <div style='background-color: #ffffff; padding: 20px; margin: 15px; border-radius: 8px; transition: transform 0.2s; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);'
             onmouseover="this.style.transform='scale(1.05)';"
             onmouseout="this.style.transform='scale(1.0)';">
            <h2 style='color: #444;'>The Great Gatsby</h2>
            <p style='color: #555;'><b>Author:</b> F. Scott Fitzgerald</p>
            <p style='color: #555;'><b>Price:</b> $10.00</p>
        </div>
    "),
    widgets.HTML(value="
        <div style='background-color: #ffffff; padding: 20px; margin: 15px; border-radius: 8px; transition: transform 0.2s; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);'
             onmouseover="this.style.transform='scale(1.05)';"
             onmouseout="this.style.transform='scale(1.0)';">
            <h2 style='color: #444;'>Pride and Prejudice</h2>
            <p style='color: #555;'><b>Author:</b> Jane Austen</p>
            <p style='color: #555;'><b>Price:</b> $15.00</p>
        </div>
    "),
]

container = widgets.VBox(book_tiles)
display(container)
"""
},
{
    "question": "Design and develop an online bookstore UI components with minimalistic theme.",
    "context": "Create the landing page for users visiting my bookstore. The landing page should display a header `Reader's Online Store`, a welcome message `Welcome to Reader's Verse` along with a catalog of books titles as title. Vertically align the sections.",
    "output": """
import ipywidgets as widgets
from IPython.display import display
# Create a header
header = widgets.HTML(value=\"\"\"
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
\"\"\")
# Create a welcome message
welcome = widgets.HTML(value=\"\"\"
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
\"\"\")
# Create book tiles
book_titles = ["The Great Gatsby", "Pride and Prejudice", "The Hobbit", "The Lord of the Rings", "Animal Farm", "Brave New World"]
book_tiles = []
for i, title in enumerate(book_titles):
    background = "#f5f5f5" if i % 2 == 0 else "#eaeaea"
    book_tiles.append(widgets.HTML(value=f\"\"\"
<div style='background-color: {background}; padding: 15px; margin: 10px; border-radius: 8px;'>
    <h2 style='color: #333;'>{title}</h2>
    <p style='color: #555;'>Author: Author Name {i+1}</p>
    <p style='color: #555;'>Price: ${(i+1)*10}.00</p>
</div>
\"\"\"))
# Create a container for the book tiles
book_container = widgets.VBox(book_tiles)
# Create a vertical box for the header, welcome message, and book container
main_container = widgets.VBox([header, welcome, book_container])
# Display the main container
display(main_container)
"""
},
{
    "question": "Refined page with Hover Effects for Enhanced User Interaction",
    "context": "Enhance the book tiles in the catalog with hover effects that change the background color, add a shadow, and slightly scale the tiles on hover. Output should also retain the Header and Welcome message aswell in the page",
    "output": """
import ipywidgets as widgets
from IPython.display import display
# Create a header
header = widgets.HTML(value=\"\"\"
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
\"\"\")
# Create a welcome message
welcome = widgets.HTML(value=\"\"\"
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
\"\"\")
# Create book tiles with hover effects
book_titles = ['The Great Gatsby', 'Pride and Prejudice', 'The Hobbit', 'The Lord of the Rings', 'Animal Farm', 'Brave New World']
book_tiles = []
for i, title in enumerate(book_titles):
    background = "#f5f5f5" if i % 2 == 0 else "#eaeaea"
    book_tiles.append(widgets.HTML(value=f\"\"\"
<div style='background-color: {background}; padding: 15px; margin: 10px; border-radius: 8px; transition: transform 0.2s; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);'
     onmouseover="this.style.transform='scale(1.05)';"
     onmouseout="this.style.transform='scale(1.0)';">
    <h2 style='color: #333;'>{title}</h2>
    <p style='color: #555;'>Author: Author Name {i+1}</p>
    <p style='color: #555;'>Price: ${(i+1)*10}.00</p>
</div>
\"\"\"))
# Create a container for the book tiles
book_container = widgets.VBox(book_tiles)

# Create a vertical box for the header, welcome message, and book container
main_container = widgets.VBox([header, welcome, book_container])
# Display the main container
display(main_container)
"""
}
]

In [6]:
# Example usage
context = "Design and develop an online bookstore UI components with minimalistic theme."
question = "Create the landing page for users visiting my bookstore. The landing page should display a header `Reader's Online Store`, a welcome message `Welcome to Reader's Verse` along with a catalog of books titles as title. Vertically align the sections."
book_titles = ["The Great Gatsby", "Pride and Prejudice", "The Hobbit", "The Lord of the Rings", "Animal Farm", "Brave New World"]
# Generate and display the UI code for the landing page
result = get_answer_using_fewshot(context, question, book_titles, examples)
print(f"Generated Code:\n{result}")

Generated Code:
```python
import ipywidgets as widgets
from IPython.display import display

# Create a header
header = widgets.HTML(value="""
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
""")

# Create a welcome message
welcome = widgets.HTML(value="""
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
""")

# Create book tiles with descriptions
book_titles = ['The Great Gatsby', 'Pride and Prejudice', 'The Hobbit', 'The Lord of the Rings', 'Animal Farm', 'Brave New World']
book_tiles = []
for i, title in enumerate(book_titles):
    background = "#f5f5f5" if i % 2 == 0 else "#eaeaea"
    description = f"A story of the Roaring Twenties, exploring themes of decadence, idealism, resistance to change, social upheaval, and excess, made famous by F. Scott Fitzgerald

In [7]:
import ipywidgets as widgets
from IPython.display import display
# Create a header
header = widgets.HTML(value="""
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
""")
# Create a welcome message
welcome = widgets.HTML(value="""
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
""")
# Create book tiles
book_titles = ["The Great Gatsby", "Pride and Prejudice", "The Hobbit", "The Lord of the Rings", "Animal Farm", "Brave New World"]
book_tiles = []
for i, title in enumerate(book_titles):
    background = "#f5f5f5" if i % 2 == 0 else "#eaeaea"
    book_tiles.append(widgets.HTML(value=f"""
<div style='background-color: {background}; padding: 15px; margin: 10px; border-radius: 8px;'>
    <h2 style='color: #333;'>{title}</h2>
    <p style='color: #555;'>Author: Author Name {i+1}</p>
    <p style='color: #555;'>Price: ${(i+1)*10}.00</p>
</div>
"""))
# Create a container for the book tiles
book_container = widgets.VBox(book_tiles)
# Create a vertical box for the header, welcome message, and book container
main_container = widgets.VBox([header, welcome, book_container])
# Display the main container
display(main_container)

In [8]:
def refine_output_with_fewshot(result, context, question, book_titles, formatted_examples):
    """
    Refines a previously generated output by applying additional user-requested changes
    using a few-shot prompt technique.

    Parameters:
    - previous_output: str, the code generated from the initial prompt.
    - context: str, the context for the refinement task.
    - question: str, the user-requested changes to be applied.
    - book_titles: list, list of book titles to include in the refined output.
    - formatted_examples: str, formatted examples of similar tasks for few-shot learning.

    Returns:
    - str, the refined Python code generated by the model.
    """

    # Format the examples for the prompt
    formatted_examples = "\n\n".join(
        f"""
        Example {i+1}:
        User Question: {example['question']}
        Context: {example['context']}
        Model Output: {example['output']}
        """
        for i, example in enumerate(examples)
    )
    prompt = f"""
    You are an experienced programmer with 15 years of experience writing full-stack applications.
    Your task is to generate high-quality Python code for a Jupyter Notebook using ipywidgets UI components
    based on the provided context and user question.

    Here are some examples of similar tasks you have completed successfully:
    {formatted_examples}

    Now, I have a Landing Page UI with basic styling created from an initial prompt. Refine the Current Page:{result} based on the additional styling and functionality changes requested in the:

    Context:
    {context}

    User Question:
    {question}
    """
    # Use the model to generate the refined code
    result = model.invoke(prompt)
    return result

In [9]:
# Example usage
context = "Refined page with Hover Effects for Enhanced User Interaction"
question = "Enhance the book tiles in the catalog with hover effects that change the background color, add a shadow, and slightly scale the tiles on hover. Output should also retain the Header and Welcome message aswell in the page"
# Generate and display the UI code for the landing page
ref_result = refine_output_with_fewshot(context, question, book_titles, examples, result)
print(f"Generated Code:\n{ref_result}")

Generated Code:
Based on your request, I will refine the existing code to include hover effects for the book tiles, changing the background color, adding a shadow, and slightly scaling the tiles on hover. The header and welcome message will remain as they are. Here's the updated code:

```python
import ipywidgets as widgets
from IPython.display import display

# Create a header
header = widgets.HTML(value="""
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
""")

# Create a welcome message
welcome = widgets.HTML(value="""
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
""")

# Create book tiles with hover effects
book_titles = ['The Great Gatsby', 'Pride and Prejudice', 'The Hobbit', 'The Lord of the Rings', 'Animal Farm', 'Brave New World']
book_tiles = []
for

In [10]:
import ipywidgets as widgets
from IPython.display import display
# Create a header
header = widgets.HTML(value="""
<div style='background-color: #333; color: white; padding: 20px; border-radius: 8px;'>
    <h1 style='text-align: center; font-family: Arial, sans-serif;'>Reader's Online Store</h1>
</div>
""")
# Create a welcome message
welcome = widgets.HTML(value="""
<div style='padding: 20px; border-radius: 8px;'>
    <h2 style='color: white; text-align: center;'>Welcome to Reader's Verse</h2>
</div>
""")
# Create book tiles with hover effects
book_titles = ['The Great Gatsby', 'Pride and Prejudice', 'The Hobbit', 'The Lord of the Rings', 'Animal Farm', 'Brave New World']
book_tiles = []
for i, title in enumerate(book_titles):
    background = "#f5f5f5" if i % 2 == 0 else "#eaeaea"
    book_tiles.append(widgets.HTML(value=f"""
<div style='background-color: {background}; padding: 15px; margin: 10px; border-radius: 8px; transition: transform 0.2s; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);'
     onmouseover="this.style.transform='scale(1.05)';"
     onmouseout="this.style.transform='scale(1.0)';">
    <h2 style='color: #333;'>{title}</h2>
    <p style='color: #555;'>Author: Author Name {i+1}</p>
    <p style='color: #555;'>Price: ${(i+1)*10}.00</p>
</div>
"""))
# Create a container for the book tiles
book_container = widgets.VBox(book_tiles)

# Create a vertical box for the header, welcome message, and book container
main_container = widgets.VBox([header, welcome, book_container])
# Display the main container
display(main_container)

SyntaxError: invalid syntax (ipython-input-711410272.py, line 1)

In [13]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
</head>
<body class="bg-gray-100">
    <header class="bg-blue-600 text-white p-6 text-center">
        <h1 class="text-4xl font-bold">Ethical Tech 101</h1>
        <p class="mt-2 text-lg">Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto p-6">
        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Mengapa Etika Teknologi Penting?</h2>
            <p class="text-gray-700">
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Topik Inti Etika Teknologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Etika dalam Cybersecurity</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Prinsip Etis Cybersecurity:</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">4 Kebiasaan Higienis Digital:</h3>
            <ul class="list-disc list-inside text-gray-700">
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor (2FA).</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Blockchain untuk Kebaikan</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Use Case "for Good":</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p class="text-gray-700">
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Insight dari Antropologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Sumber Daya</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Kontak</h2>
            <p class="text-gray-700">
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer class="bg-gray-800 text-white p-6 text-center mt-8">
        <p>&copy; 2023 Ethical Tech 101</p>
    </footer>
</body>
</html>

SyntaxError: invalid decimal literal (ipython-input-1229637628.py, line 43)

In [14]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
</head>
<body class="bg-gray-100">
    <header class="bg-blue-600 text-white p-6 text-center">
        <h1 class="text-4xl font-bold">Ethical Tech 101</h1>
        <p class="mt-2 text-lg">Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto p-6">
        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Mengapa Etika Teknologi Penting?</h2>
            <p class="text-gray-700">
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Topik Inti Etika Teknologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Etika dalam Cybersecurity</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Prinsip Etis Cybersecurity:</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">4 Kebiasaan Higienis Digital:</h3>
            <ul class="list-disc list-inside text-gray-700">
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Blockchain untuk Kebaikan</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Use Case "for Good":</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p class="text-gray-700">
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Insight dari Antropologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Sumber Daya</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Kontak</h2>
            <p class="text-gray-700">
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer class="bg-gray-800 text-white p-6 text-center mt-8">
        <p>&copy; 2023 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [15]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            color: #333; /* Darker text for better contrast */
            line-height: 1.6;
        }
        h1, h2, h3 {
            color: #1a202c; /* Even darker headings */
        }
        .container {
            background-color: #fff; /* White background for the main content area */
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1); /* Subtle shadow for definition */
        }
        section {
            margin-bottom: 2rem; /* Increased spacing between sections */
            padding-bottom: 1.5rem;
            border-bottom: 1px solid #e2e8f0; /* Light border to separate sections */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }
        ul {
            margin-top: 1rem;
        }
        li {
            margin-bottom: 0.5rem;
        }
        a {
            color: #3182ce; /* Brighter link color */
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
    </style>
</head>
<body class="bg-gray-100">
    <header class="bg-blue-600 text-white p-6 text-center">
        <h1 class="text-4xl font-bold">Ethical Tech 101</h1>
        <p class="mt-2 text-lg">Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto p-6">
        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Mengapa Etika Teknologi Penting?</h2>
            <p class="text-gray-700">
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Topik Inti Etika Teknologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Etika dalam Cybersecurity</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Prinsip Etis Cybersecurity:</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">4 Kebiasaan Higienis Digital:</h3>
            <ul class="list-disc list-inside text-gray-700">
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Blockchain untuk Kebaikan</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Use Case "for Good":</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p class="text-gray-700">
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Insight dari Antropologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Sumber Daya</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Kontak</h2>
            <p class="text-gray-700">
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer class="bg-gray-800 text-white p-6 text-center mt-8">
        <p>&copy; 2023 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [16]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            color: #333; /* Darker text for better contrast */
            line-height: 1.6;
        }
        h1, h2, h3 {
            color: #1a202c; /* Even darker headings */
        }
        /* Styling for sub-section titles */
        h3 {
            color: #0056b3; /* A shade of blue for sub-titles */
            margin-top: 1.5rem; /* Add some space above subheadings */
        }
        .container {
            background-color: #fff; /* White background for the main content area */
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1); /* Subtle shadow for definition */
        }
        section {
            margin-bottom: 2rem; /* Increased spacing between sections */
            padding-bottom: 1.5rem;
            border-bottom: 1px solid #e2e8f0; /* Light border to separate sections */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }
        ul {
            margin-top: 1rem;
        }
        li {
            margin-bottom: 0.5rem;
        }
        a {
            color: #3182ce; /* Brighter link color */
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
        /* Styling for the main content text */
        main p, main li {
            font-size: 1.1rem; /* Increase font size for main content paragraphs and list items */
        }
        /* Styling for the footer copyright text */
        footer p {
            font-size: 1rem; /* Revert copyright font size to default or desired smaller size */
            color: #cccccc; /* Lighter color for copyright */
        }
    </style>
</head>
<body class="bg-gray-100">
    <header class="bg-blue-600 text-white p-6 text-center">
        <h1 class="text-4xl font-bold">Ethical Tech 101</h1>
        <p class="mt-2 text-lg">Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto p-6">
        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Mengapa Etika Teknologi Penting?</h2>
            <p class="text-gray-700">
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Topik Inti Etika Teknologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Etika dalam Cybersecurity</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Prinsip Etis Cybersecurity:</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">4 Kebiasaan Higienis Digital:</h3>
            <ul class="list-disc list-inside text-gray-700">
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Blockchain untuk Kebaikan</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Use Case "for Good":</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p class="text-gray-700">
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Insight dari Antropologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Sumber Daya</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Kontak</h2>
            <p class="text-gray-700">
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer class="bg-gray-800 text-white p-6 text-center mt-8">
        <p>&copy; 2024 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [17]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            color: #333; /* Darker text for better contrast */
            line-height: 1.6;
            background-color: #f0f9ff; /* Light blue background for a touch of color */
        }
        h1, h2, h3 {
            color: #1a202c; /* Even darker headings */
        }
        /* Styling for sub-section titles */
        h3 {
            color: #0056b3; /* A shade of blue for sub-titles */
            margin-top: 1.5rem; /* Add some space above subheadings */
        }
        .container {
            background-color: #ffffff; /* White background for the main content area */
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1); /* Subtle shadow for definition */
        }
        section {
            margin-bottom: 2rem; /* Increased spacing between sections */
            padding-bottom: 1.5rem;
            border-bottom: 1px solid #e2e8f0; /* Light border to separate sections */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }
        ul {
            margin-top: 1rem;
        }
        li {
            margin-bottom: 0.5rem;
        }
        a {
            color: #3182ce; /* Brighter link color */
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
        /* Styling for the main content text */
        main p, main li {
            font-size: 1.1rem; /* Increase font size for main content paragraphs and list items */
        }
        /* Styling for the footer copyright text */
        footer p {
            font-size: 1rem; /* Revert copyright font size to default or desired smaller size */
            color: #cccccc; /* Lighter color for copyright */
        }
        header {
            background-color: #1e3a8a; /* Darker blue for header */
        }
        footer {
            background-color: #1f2937; /* Dark gray for footer */
        }
    </style>
</head>
<body class="bg-gray-100">
    <header class="bg-blue-600 text-white p-6 text-center">
        <h1 class="text-4xl font-bold">Ethical Tech 101</h1>
        <p class="mt-2 text-lg">Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto p-6">
        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Mengapa Etika Teknologi Penting?</h2>
            <p class="text-gray-700">
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Topik Inti Etika Teknologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Etika dalam Cybersecurity</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Prinsip Etis Cybersecurity:</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">4 Kebiasaan Higienis Digital:</h3>
            <ul class="list-disc list-inside text-gray-700">
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Blockchain untuk Kebaikan</h2>
            <h3 class="text-xl font-semibold text-gray-700 mb-2">3 Use Case "for Good":</h3>
            <ul class="list-disc list-inside text-gray-700 mb-4">
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p class="text-gray-700">
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Insight dari Antropologi</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section class="mb-8">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Sumber Daya</h2>
            <ul class="list-disc list-inside text-gray-700">
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Kontak</h2>
            <p class="text-gray-700">
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer class="bg-gray-800 text-white p-6 text-center mt-8">
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [18]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            font-family: 'Arial', sans-serif; /* Using a common sans-serif font */
            line-height: 1.8; /* Increased line height for better readability */
            background-color: #f4f7f6; /* A slightly warmer light background */
            color: #333; /* Dark text */
        }
        header {
            background: linear-gradient to right, #4a0e9b, #2c3e50; /* Gradient header */
            color: #e0e0e0; /* Lighter text in header */
            padding: 30px 20px; /* More padding */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Stronger shadow for header */
        }
        header h1 {
            font-size: 3rem; /* Larger main title */
            font-weight: 700;
            color: #ffffff; /* White title */
        }
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Lighter subtitle */
        }

        .container {
            background-color: #ffffff; /* White background for main content */
            padding: 30px; /* More padding */
            border-radius: 12px; /* More rounded corners */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* More pronounced shadow */
            margin-top: 20px; /* Space below header */
        }

        section {
            margin-bottom: 3rem; /* More space between sections */
            padding-bottom: 2rem;
            border-bottom: 1px solid #e0e0e0; /* Lighter border */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        h2 {
            font-size: 2.2rem; /* Larger section titles */
            font-weight: 600;
            color: #1a202c; /* Dark color for section titles */
            margin-bottom: 1.5rem; /* More space below section titles */
            border-left: 5px solid #4a0e9b; /* Accent color border */
            padding-left: 10px;
        }

        h3 {
            font-size: 1.6rem; /* Larger sub-titles */
            font-weight: 600;
            color: #0056b3; /* Blue color for sub-titles */
            margin-top: 2rem; /* More space above subheadings */
            margin-bottom: 1rem; /* More space below subheadings */
        }

        main p, main li {
            font-size: 1.15rem; /* Slightly larger font size for content text */
            color: #555; /* Slightly lighter color for body text */
        }

        ul {
            margin-top: 1.5rem; /* More space above lists */
        }

        li {
            margin-bottom: 0.8rem; /* More space between list items */
        }

        a {
            color: #007bff; /* Standard blue for links */
            text-decoration: none;
            transition: color 0.3s ease; /* Smooth color transition on hover */
        }
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Darker blue on hover */
        }

        footer {
            background-color: #34495e; /* Darker footer color */
            color: #b0b0b0; /* Lighter copyright text */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* More space above footer */
        }
        footer p {
            font-size: 1rem; /* Standard font size for copyright */
        }
    </style>
</head>
<body>
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto">
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkepan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section>
            <h2>Sumber Daya</h2>
            <ul>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [19]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            font-family: 'Arial', sans-serif; /* Using a common sans-serif font */
            line-height: 1.8; /* Increased line height for better readability */
            background-color: #f4f7f6; /* A slightly warmer light background */
            color: #333; /* Dark text */
        }
        header {
            background: linear-gradient(to right, #4a0e9b, #2c3e50); /* Gradient header */
            color: #e0e0e0; /* Lighter text in header */
            padding: 30px 20px; /* More padding */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Stronger shadow for header */
        }
        header h1 {
            font-size: 3rem; /* Larger main title */
            font-weight: 700;
            color: #ffffff; /* White title for clarity */
        }
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Lighter subtitle */
        }

        .container {
            background-color: #e9ecef; /* Light gray background for main content */
            padding: 30px; /* More padding */
            border-radius: 12px; /* More rounded corners */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* More pronounced shadow */
            margin-top: 20px; /* Space below header */
        }

        section {
            margin-bottom: 3rem; /* More space between sections */
            padding-bottom: 2rem;
            border-bottom: 1px solid #ced4da; /* Lighter border */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        h2 {
            font-size: 2.2rem; /* Larger section titles */
            font-weight: 600;
            color: #1a202c; /* Dark color for section titles */
            margin-bottom: 1.5rem; /* More space below section titles */
            border-left: 5px solid #4a0e9b; /* Accent color border */
            padding-left: 10px;
        }

        h3 {
            font-size: 1.6rem; /* Larger sub-titles */
            font-weight: 600;
            color: #0056b3; /* Blue color for sub-titles */
            margin-top: 2rem; /* More space above subheadings */
            margin-bottom: 1rem; /* More space below subheadings */
        }

        main p, main li {
            font-size: 1.25rem; /* Larger font size for content text */
            font-weight: bold; /* Make content text bold */
            color: #555; /* Slightly lighter color for body text */
        }

        ul {
            margin-top: 1.5rem; /* More space above lists */
        }

        li {
            margin-bottom: 0.8rem; /* More space between list items */
        }

        a {
            color: #007bff; /* Standard blue for links */
            text-decoration: none;
            transition: color 0.3s ease; /* Smooth color transition on hover */
        }
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Darker blue on hover */
        }

        footer {
            background-color: #34495e; /* Darker footer color */
            color: #b0b0b0; /* Lighter copyright text */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* More space above footer */
        }
        footer p {
            font-size: 1rem; /* Standard font size for copyright */
        }
    </style>
</head>
<body>
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto">
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section>
            <h2>Sumber Daya</h2>
            <ul>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 1</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 2</a></li>
                <li><a href="#" class="text-blue-600 hover:underline">Link Sumber Kredibel 3</a></li>
            </ul>
        </section>

        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [20]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            font-family: 'Arial', sans-serif; /* Using a common sans-serif font */
            line-height: 1.8; /* Increased line height for better readability */
            background-color: #f4f7f6; /* A slightly warmer light background */
            color: #333; /* Dark text */
        }
        header {
            background: linear-gradient(to right, #4a0e9b, #2c3e50); /* Gradient header */
            color: #e0e0e0; /* Lighter text in header */
            padding: 30px 20px; /* More padding */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Stronger shadow for header */
        }
        header h1 {
            font-size: 3rem; /* Larger main title */
            font-weight: 700;
            color: #ffffff; /* White title for clarity */
        }
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Lighter subtitle */
        }

        .container {
            background-color: #e9ecef; /* Light gray background for main content */
            padding: 30px; /* More padding */
            border-radius: 12px; /* More rounded corners */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* More pronounced shadow */
            margin-top: 20px; /* Space below header */
        }

        section {
            margin-bottom: 3rem; /* More space between sections */
            padding-bottom: 2rem;
            border-bottom: 1px solid #ced4da; /* Lighter border */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        h2 {
            font-size: 2.2rem; /* Larger section titles */
            font-weight: 600;
            color: #1a202c; /* Dark color for section titles */
            margin-bottom: 1.5rem; /* More space below section titles */
            border-left: 5px solid #4a0e9b; /* Accent color border */
            padding-left: 10px;
        }

        h3 {
            font-size: 1.6rem; /* Larger sub-titles */
            font-weight: 600;
            color: #0056b3; /* Blue color for sub-titles */
            margin-top: 2rem; /* More space above subheadings */
            margin-bottom: 1rem; /* More space below subheadings */
        }

        main p, main li {
            font-size: 1.25rem; /* Larger font size for content text */
            font-weight: bold; /* Make content text bold */
            color: #555; /* Slightly lighter color for body text */
        }

        ul {
            margin-top: 1.5rem; /* More space above lists */
        }

        li {
            margin-bottom: 0.8rem; /* More space between list items */
        }

        a {
            color: #007bff; /* Standard blue for links */
            text-decoration: none;
            transition: color 0.3s ease; /* Smooth color transition on hover */
        }
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Darker blue on hover */
        }

        footer {
            background-color: #34495e; /* Darker footer color */
            color: #b0b0b0; /* Lighter copyright text */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* More space above footer */
        }
        footer p {
            font-size: 1rem; /* Standard font size for copyright */
        }
    </style>
</head>
<body>
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto">
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section>
            <h2>Referensi</h2>
            <ul>
                <li><a href="https://www.example.com/ethical-tech-resource-1" class="text-blue-600 hover:underline">Ethical Technology Overview (Example Link 1)</a></li>
                <li><a href="https://www.example.com/cybersecurity-ethics-resource" class="text-blue-600 hover:underline">Cybersecurity Ethics (Example Link 2)</a></li>
                <li><a href="https://www.example.com/blockchain-social-impact" class="text-blue-600 hover:underline">Blockchain and Society (Example Link 3)</a></li>
                <li><a href="https://www.example.com/anthropology-technology" class="text-blue-600 hover:underline">Anthropology of Technology (Example Link 4)</a></li>
            </ul>
        </section>

        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [21]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            font-family: 'Arial', sans-serif; /* Using a common sans-serif font */
            line-height: 1.8; /* Increased line height for better readability */
            background-color: #f4f7f6; /* A slightly warmer light background */
            color: #333; /* Dark text */
        }
        header {
            background: linear-gradient(to right, #4a0e9b, #2c3e50); /* Gradient header */
            color: #e0e0e0; /* Lighter text in header */
            padding: 30px 20px; /* More padding */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Stronger shadow for header */
        }
        header h1 {
            font-size: 3rem; /* Larger main title */
            font-weight: 700;
            color: #ffffff; /* White title for clarity */
        }
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Lighter subtitle */
        }

        .container {
            background-color: #e9ecef; /* Light gray background for main content */
            padding: 30px; /* More padding */
            border-radius: 12px; /* More rounded corners */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* More pronounced shadow */
            margin-top: 20px; /* Space below header */
        }

        section {
            margin-bottom: 3rem; /* More space between sections */
            padding-bottom: 2rem;
            border-bottom: 1px solid #ced4da; /* Lighter border */
        }
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        h2 {
            font-size: 2.2rem; /* Larger section titles */
            font-weight: 600;
            color: #1a202c; /* Dark color for section titles */
            margin-bottom: 1.5rem; /* More space below section titles */
            border-left: 5px solid #4a0e9b; /* Accent color border */
            padding-left: 10px;
        }

        h3 {
            font-size: 1.6rem; /* Larger sub-titles */
            font-weight: 600;
            color: #0056b3; /* Blue color for sub-titles */
            margin-top: 2rem; /* More space above subheadings */
            margin-bottom: 1rem; /* More space below subheadings */
        }

        main p, main li {
            font-size: 1.25rem; /* Larger font size for content text */
            font-weight: bold; /* Make content text bold */
            color: #555; /* Slightly lighter color for body text */
        }

        ul {
            margin-top: 1.5rem; /* More space above lists */
        }

        li {
            margin-bottom: 0.8rem; /* More space between list items */
        }

        a {
            color: #007bff; /* Standard blue for links */
            text-decoration: none;
            transition: color 0.3s ease; /* Smooth color transition on hover */
        }
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Darker blue on hover */
        }

        footer {
            background-color: #34495e; /* Darker footer color */
            color: #b0b0b0; /* Lighter copyright text */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* More space above footer */
        }
        footer p {
            font-size: 1rem; /* Standard font size for copyright */
        }
    </style>
</head>
<body>
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto">
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section>
            <h2>Referensi</h2>
            <ul>
                <li><a href="https://www.sciencedirect.com/topics/social-sciences/ethics-of-technology" class="text-blue-600 hover:underline">Ethical Technology Overview (ScienceDirect)</a></li>
                <li><a href="https://link.springer.com/chapter/10.1007/978-3-031-04036-8_9" class="text-blue-600 hover:underline">Cybersecurity Ethics (Springer Link)</a></li>
                <li><a href="https://www.sciencedirect.com/science/article/abs/pii/S0308596124000156" class="text-blue-600 hover:underline">Blockchain and Society (ScienceDirect)</a></li>
                <li><a href="https://www.anthroencyclopedia.com/entry/technology" class="text-blue-600 hover:underline">Anthropology of Technology (AnthroEncyclopedia)</a></li>
            </ul>
        </section>

        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [22]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body {
            font-family: 'Arial', sans-serif; /* Using a common sans-serif font */
            line-height: 1.8; /* Increased line height for better readability */
            background-color: #f4f7f6; /* A slightly warmer light background */
            color: #333; /* Dark text */
        }
        header {
            background: linear-gradient(to right, #4a0e9b, #2c3e50); /* Gradient header */
            color: #e0e0e0; /* Lighter text in header */
            padding: 30px 20px; /* More padding */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Stronger shadow for header */
        }
        header h1 {
            font-size: 3rem; /* Larger main title */
            font-weight: 700;
            color: #ffffff; /* White title for clarity */
        }
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Lighter subtitle */
        }

        .container {
            background-color: #e9ecef; /* Light gray background for main content */
            padding: 30px; /* More padding */
            border-radius: 12px; /* More rounded corners */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* More pronounced shadow */
            margin-top: 20px; /* Space below header */
        }

        section {
            margin-bottom: 3rem; /* More space between sections */
            padding-bottom: 2rem;
            border-bottom: 1px solid #ced4da; /* Lighter border */
            opacity: 0; /* Start hidden for animation */
            transform: translateY(20px); /* Start slightly below */
            animation: fadeIn 0.8s ease-out forwards; /* Apply animation */
        }
        section:nth-child(1) { animation-delay: 0.2s; } /* Stagger animation */
        section:nth-child(2) { animation-delay: 0.4s; }
        section:nth-child(3) { animation-delay: 0.6s; }
        section:nth-child(4) { animation-delay: 0.8s; }
        section:nth-child(5) { animation-delay: 1.0s; }
        section:nth-child(6) { animation-delay: 1.2s; }
        section:nth-child(7) { animation-delay: 1.4s; }


        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        h2 {
            font-size: 2.2rem; /* Larger section titles */
            font-weight: 600;
            color: #1a202c; /* Dark color for section titles */
            margin-bottom: 1.5rem; /* More space below section titles */
            border-left: 5px solid #4a0e9b; /* Accent color border */
            padding-left: 10px;
        }

        h3 {
            font-size: 1.6rem; /* Larger sub-titles */
            font-weight: 600;
            color: #0056b3; /* Blue color for sub-titles */
            margin-top: 2rem; /* More space above subheadings */
            margin-bottom: 1rem; /* More space below subheadings */
        }

        main p, main li {
            font-size: 1.25rem; /* Larger font size for content text */
            font-weight: bold; /* Make content text bold */
            color: #555; /* Slightly lighter color for body text */
        }

        ul {
            margin-top: 1.5rem; /* More space above lists */
        }

        li {
            margin-bottom: 0.8rem; /* More space between list items */
        }

        a {
            color: #007bff; /* Standard blue for links */
            text-decoration: none;
            transition: color 0.3s ease; /* Smooth color transition on hover */
        }
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Darker blue on hover */
        }

        footer {
            background-color: #34495e; /* Darker footer color */
            color: #b0b0b0; /* Lighter copyright text */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* More space above footer */
        }
        footer p {
            font-size: 1rem; /* Standard font size for copyright */
        }

        /* Keyframes for fade-in animation */
        @keyframes fadeIn {
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }
    </style>
</head>
<body>
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <main class="container mx-auto">
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <section>
            <h2>Referensi</h2>
            <ul>
                <li><a href="https://www.sciencedirect.com/topics/social-sciences/ethics-of-technology" class="text-blue-600 hover:underline">Ethical Technology Overview (ScienceDirect)</a></li>
                <li><a href="https://link.springer.com/chapter/10.1007/978-3-031-04036-8_9" class="text-blue-600 hover:underline">Cybersecurity Ethics (Springer Link)</a></li>
                <li><a href="https://www.sciencedirect.com/science/article/abs/pii/S0308596124000156" class="text-blue-600 hover:underline">Blockchain and Society (ScienceDirect)</a></li>
                <li><a href="https://www.anthroencyclopedia.com/entry/technology" class="text-blue-600 hover:underline">Anthropology of Technology (AnthroEncyclopedia)</a></li>
            </ul>
        </section>

        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

display(HTML(html_content))

In [23]:
from IPython.display import HTML

# Konten HTML untuk halaman landing Ethical Tech 101
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ethical Tech 101</title>
    <!-- Tautan ke Tailwind CSS untuk styling -->
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        /* Styling dasar untuk body: font, tinggi baris, dan latar belakang */
        body {
            font-family: 'Arial', sans-serif; /* Menggunakan font sans-serif umum */
            line-height: 1.8; /* Meningkatkan tinggi baris untuk keterbacaan yang lebih baik */
            background-color: #f4f7f6; /* Latar belakang terang yang sedikit hangat */
            color: #333; /* Teks gelap */
        }
        /* Styling untuk judul utama (h1, h2, h3) */
        h1, h2, h3 {
            color: #1a202c; /* Judul yang lebih gelap */
        }
        /* Styling untuk bagian header */
        header {
            background: linear-gradient(to right, #4a0e9b, #2c3e50); /* Latar belakang gradien untuk header */
            color: #e0e0e0; /* Teks lebih terang di header */
            padding: 30px 20px; /* Padding lebih banyak */
            text-align: center;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); /* Bayangan lebih kuat untuk header */
        }
        /* Styling untuk judul utama di header */
        header h1 {
            font-size: 3rem; /* Ukuran judul utama lebih besar */
            font-weight: 700;
            color: #ffffff; /* Judul putih agar jelas */
        }
        /* Styling untuk subtitle di header */
        header p {
            font-size: 1.2rem;
            margin-top: 10px;
            color: #cccccc; /* Subtitle lebih terang */
        }

        /* Styling untuk container konten utama */
        .container {
            background-color: #e9ecef; /* Latar belakang abu-abu terang untuk konten utama */
            padding: 30px; /* Padding lebih banyak */
            border-radius: 12px; /* Sudut lebih membulat */
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15); /* Bayangan lebih jelas */
            margin-top: 20px; /* Ruang di bawah header */
        }

        /* Styling untuk setiap bagian (section) dalam konten utama */
        section {
            margin-bottom: 3rem; /* Ruang lebih banyak antar bagian */
            padding-bottom: 2rem;
            border-bottom: 1px solid #ced4da; /* Garis bawah lebih terang */
            opacity: 0; /* Mulai tersembunyi untuk animasi */
            transform: translateY(20px); /* Mulai sedikit di bawah */
            animation: fadeIn 0.8s ease-out forwards; /* Terapkan animasi */
        }
        /* Penundaan animasi yang berbeda untuk setiap bagian (staggering) */
        section:nth-child(1) { animation-delay: 0.2s; }
        section:nth-child(2) { animation-delay: 0.4s; }
        section:nth-child(3) { animation-delay: 0.6s; }
        section:nth-child(4) { animation-delay: 0.8s; }
        section:nth-child(5) { animation-delay: 1.0s; }
        section:nth-child(6) { animation-delay: 1.2s; }
        section:nth-child(7) { animation-delay: 1.4s; }


        /* Menghapus border dan margin untuk bagian terakhir */
        section:last-child {
            border-bottom: none;
            margin-bottom: 0;
            padding-bottom: 0;
        }

        /* Styling untuk judul bagian (h2) */
        h2 {
            font-size: 2.2rem; /* Ukuran judul bagian lebih besar */
            font-weight: 600;
            color: #1a202c; /* Warna gelap untuk judul bagian */
            margin-bottom: 1.5rem; /* Ruang lebih banyak di bawah judul bagian */
            border-left: 5px solid #4a0e9b; /* Border warna aksen */
            padding-left: 10px;
        }

        /* Styling untuk judul sub-bagian (h3) */
        h3 {
            font-size: 1.6rem; /* Ukuran sub-judul lebih besar */
            font-weight: 600;
            color: #0056b3; /* Warna biru untuk sub-judul */
            margin-top: 2rem; /* Ruang lebih banyak di atas sub-judul */
            margin-bottom: 1rem; /* Ruang lebih banyak di bawah sub-judul */
        }

        /* Styling untuk paragraf dan item list di konten utama */
        main p, main li {
            font-size: 1.25rem; /* Ukuran font lebih besar untuk teks konten */
            font-weight: bold; /* Membuat teks konten menjadi bold */
            color: #555; /* Warna sedikit lebih terang untuk teks body */
        }

        /* Styling untuk daftar tidak berurutan (unordered lists) */
        ul {
            margin-top: 1.5rem; /* Ruang lebih banyak di atas daftar */
        }

        /* Styling untuk item daftar (list items) */
        li {
            margin-bottom: 0.8rem; /* Ruang lebih banyak antar item daftar */
        }

        /* Styling untuk tautan (links) */
        a {
            color: #007bff; /* Biru standar untuk tautan */
            text-decoration: none;
            transition: color 0.3s ease; /* Transisi warna halus saat hover */
        }
        /* Styling untuk tautan saat di-hover */
        a:hover {
            text-decoration: underline;
            color: #0056b3; /* Biru lebih gelap saat hover */
        }

        /* Styling untuk footer */
        footer {
            background-color: #34495e; /* Warna footer lebih gelap */
            color: #b0b0b0; /* Teks copyright lebih terang */
            padding: 20px;
            text-align: center;
            margin-top: 40px; /* Ruang lebih banyak di atas footer */
        }
        /* Styling untuk teks copyright di footer */
        footer p {
            font-size: 1rem; /* Ukuran font standar untuk copyright */
        }

        /* Keyframes untuk animasi fade-in */
        @keyframes fadeIn {
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }
    </style>
</head>
<body>
    <!-- Bagian Header -->
    <header>
        <h1>Ethical Tech 101</h1>
        <p>Teknologi sebagai alat untuk kemanusiaan, bukan sebaliknya.</p>
    </header>

    <!-- Area konten utama -->
    <main class="container mx-auto">
        <!-- Bagian: Mengapa Etika Teknologi Penting? -->
        <section>
            <h2>Mengapa Etika Teknologi Penting?</h2>
            <p>
                Dari sudut pandang antropologi, teknologi bukan sekadar alat, melainkan pembentuk budaya dan relasi sosial. Memahami etika teknologi berarti menjaga agar inovasi digital selaras dengan nilai-nilai kemanusiaan, mencegah disrupsi yang merugikan, dan memastikan teknologi melayani keragaman manusia, bukan mendominasinya. Ini adalah langkah krusial untuk masa depan digital yang inklusif dan berkelanjutan.
            </p>
        </section>

        <!-- Bagian: Topik Inti Etika Teknologi -->
        <section>
            <h2>Topik Inti Etika Teknologi</h2>
            <ul>
                <li><strong>Privasi Data:</strong> Hak individu atas informasi pribadi di era digital.</li>
                <li><strong>Bias dalam AI:</strong> Bagaimana algoritma dapat mencerminkan atau memperkuat prasangka sosial.</li>
                <li><strong>Dampak Sosial Teknologi:</strong> Pengaruh teknologi terhadap interaksi manusia, komunitas, dan struktur kekuasaan.</li>
            </ul>
        </section>

        <!-- Bagian: Etika dalam Cybersecurity -->
        <section>
            <h2>Etika dalam Cybersecurity</h2>
            <h3>3 Prinsip Etis Cybersecurity:</h3>
            <ul>
                <li>Integritas: Menjaga keakuratan dan kelengkapan informasi.</li>
                <li>Kerahasiaan: Melindungi data dari akses tidak sah.</li>
                <li>Ketersediaan: Memastikan sistem dan data dapat diakses saat dibutuhkan oleh pihak yang berwenang.</li>
            </ul>
            <h3>4 Kebiasaan Higienis Digital:</h3>
            <ul>
                <li>Gunakan kata sandi yang kuat dan unik.</li>
                <li>Aktifkan otentikasi dua faktor.</li>
                <li>Waspadai email phishing dan tautan mencurigakan.</li>
                <li>Perbarui perangkat lunak secara teratur.</li>
            </ul>
        </section>

        <!-- Bagian: Blockchain untuk Kebaikan -->
        <section>
            <h2>Blockchain untuk Kebaikan</h2>
            <h3>3 Use Case "for Good":</h3>
            <ul>
                <li>Pelacakan rantai pasok yang transparan untuk produk berkelanjutan.</li>
                <li>Identitas digital yang aman untuk pengungsi dan populasi rentan.</li>
                <li>Sistem voting yang aman dan tidak dapat dimanipulasi.</li>
            </ul>
            <p>
                Namun, potensi blockchain harus diimbangi dengan tata kelola yang matang dan pemahaman konteks sosial agar tidak menciptakan kesenjangan baru.
            </p>
        </section>

        <!-- Bagian: Insight dari Antropologi -->
        <section>
            <h2>Insight dari Antropologi</h2>
            <ul>
                <li><strong>Praktik Harian:</strong> Amati bagaimana teknologi mengubah ritual sosial, komunikasi keluarga, atau cara kita bekerja. Kesadaran ini membantu kita menggunakan teknologi dengan lebih mindful.</li>
                <li><strong>Desain Peka Budaya:</strong> Pertimbangkan latar belakang budaya dan nilai-nilai lokal saat merancang atau mengimplementasikan teknologi agar relevan dan diterima masyarakat.</li>
            </ul>
        </section>

        <!-- Bagian: Referensi -->
        <section>
            <h2>Referensi</h2>
            <ul>
                <li><a href="https://www.sciencedirect.com/topics/social-sciences/ethics-of-technology" class="text-blue-600 hover:underline">Ethical Technology Overview (ScienceDirect)</a></li>
                <li><a href="https://link.springer.com/chapter/10.1007/978-3-031-04036-8_9" class="text-blue-600 hover:underline">Cybersecurity Ethics (Springer Link)</a></li>
                <li><a href="https://www.sciencedirect.com/science/article/abs/pii/S0308596124000156" class="text-blue-600 hover:underline">Blockchain and Society (ScienceDirect)</a></li>
                <li><a href="https://www.anthroencyclopedia.com/entry/technology" class="text-blue-600 hover:underline">Anthropology of Technology (AnthroEncyclopedia)</a></li>
            </ul>
        </section>

        <!-- Bagian: Kontak -->
        <section>
            <h2>Kontak</h2>
            <p>
                Untuk pertanyaan lebih lanjut, hubungi kami di <a href="mailto:natatether@proton.me" class="text-blue-600 hover:underline">natatether@proton.me</a>.
            </p>
        </section>
    </main>

    <!-- Bagian Footer -->
    <footer>
        <p>&copy; 2025 Ethical Tech 101</p>
    </footer>
</body>
</html>
"""

# Menampilkan konten HTML di output notebook
display(HTML(html_content))

In [24]:
# Ethical Tech 101

**Description:**
Ini adalah landing page edukasi singkat tentang etika teknologi, menghubungkan cybersecurity, blockchain, dan antropologi. Halaman ini bertujuan untuk memberikan pemahaman dasar tentang pentingnya pertimbangan etis dalam pengembangan dan penggunaan teknologi di era digital.

**Technologies Used:**
*   HTML
*   Tailwind CSS
*   Netlify (untuk deployment)

**Features:**
*   **Hero Section:** Visi singkat tentang teknologi untuk kemanusiaan.
*   **About Section:** Penjelasan mengapa etika teknologi penting dari sudut pandang antropologi.
*   **Cybersecurity Section:** Prinsip etis dan kebiasaan higienis dalam keamanan siber.
*   **Blockchain Section:** Use case "for good" dan pertimbangan tata kelola blockchain.
*   **Anthropology Section:** Insight aplikatif dari sudut pandang antropologi terkait teknologi.
*   **References Section:** Tautan ke sumber-sumber kredibel terkait topik yang dibahas.
*   **Contact Section:** Informasi kontak untuk pertanyaan lebih lanjut.

**Setup Instructions:**
1.  Clone repository ini ke komputer lokal Anda:

SyntaxError: invalid syntax (ipython-input-2921270280.py, line 3)

# Ethical Tech 101

**Description:**
Ini adalah landing page edukasi singkat tentang etika teknologi, menghubungkan cybersecurity, blockchain, dan antropologi. Halaman ini bertujuan untuk memberikan pemahaman dasar tentang pentingnya pertimbangan etis dalam pengembangan dan penggunaan teknologi di era digital.

**Technologies Used:**
*   HTML
*   Tailwind CSS
*   Netlify (untuk deployment)

**Features:**
*   **Hero Section:** Visi singkat tentang teknologi untuk kemanusiaan.
*   **About Section:** Penjelasan mengapa etika teknologi penting dari sudut pandang antropologi.
*   **Cybersecurity Section:** Prinsip etis dan kebiasaan higienis dalam keamanan siber.
*   **Blockchain Section:** Use case "for good" dan pertimbangan tata kelola blockchain.
*   **Anthropology Section:** Insight aplikatif dari sudut pandang antropologi terkait teknologi.
*   **References Section:** Tautan ke sumber-sumber kredibel terkait topik yang dibahas.
*   **Contact Section:** Informasi kontak untuk pertanyaan lebih lanjut.

**Setup Instructions:**
1.  Clone repository ini ke komputer lokal Anda: